## Fama Macbeth Regression

In [91]:
import wrds
import pandas as pd
import numpy as np

from statsmodels.regression.rolling import RollingOLS
import statsmodels.formula.api as smf
import scipy.stats as stats

import linearmodels as lm

## 어지원 조교님 세션

### Load FF tables

In [31]:
db = wrds.Connection(
    wrds_username="jaewrdskaist",
    wrds_password="hisoWRDS23@#"
)

Loading library list...
Done


In [32]:
db.list_tables(library='ff')

['factors_china',
 'factors_daily',
 'factors_monthly',
 'fivefactors_daily',
 'fivefactors_monthly',
 'industry12',
 'industry48',
 'liq_ps',
 'liq_sadka',
 'portfolios',
 'portfolios25',
 'portfolios_d']

### 25 size, value vw portfolios

In [33]:
portfolio = db.get_table(table='portfolios25', library='ff')
portfolio = portfolio[3:]

In [34]:
portfolio['date'] = pd.to_datetime(portfolio['date'], format='%Y-%m-%d')
date = portfolio['date'] - pd.tseries.offsets.MonthEnd()
portfolio = portfolio.iloc[:, 0:25]

In [35]:
date

3             NaT
4             NaT
5             NaT
6             NaT
7             NaT
          ...    
1185   2024-07-31
1186   2024-08-31
1187   2024-09-30
1188   2024-10-31
1189   2024-11-30
Name: date, Length: 1187, dtype: datetime64[ns]

In [36]:
portfolio['date'] = date

In [37]:
portfolio = pd.melt(portfolio, id_vars=['date'])

### Monthly FF3 Factors & Risk Free Rate

In [38]:
monthlyff = db.get_table(table='factors_monthly', library='ff')[['date', 'mktrf', 'hml', 'smb', 'rf']]
monthlyff['date'] = pd.to_datetime(monthlyff['date'], format='%Y-%m-%d')
monthlyff['date'] = monthlyff['date'] - pd.tseries.offsets.MonthEnd()

In [39]:
portfolio.dropna(subset=['date'], inplace=True)

### Final Dataset

In [40]:
data = pd.merge(portfolio, monthlyff, on='date', how='left')
data['exret'] = data['value'] - data['rf']

In [41]:
data.head()

,date,variable,value,mktrf,hml,smb,rf,exret
0,1926-06-30,s1b1_vwret,0.0583,0.0289,-0.0239,-0.0255,0.0022,0.0561
1,1926-07-31,s1b1_vwret,-0.0202,0.0264,0.0381,-0.0114,0.0025,-0.0227
2,1926-08-31,s1b1_vwret,-0.0483,0.0038,0.0005,-0.0136,0.0023,-0.0506
3,1926-09-30,s1b1_vwret,-0.0936,-0.0327,0.0082,-0.0014,0.0032,-0.0968
4,1926-10-31,s1b1_vwret,0.0559,0.0254,-0.0061,-0.0011,0.0031,0.0528


## Method 1. 조교님 방법

### Rolling FF3F beta

$ r_{i,t}^{e} = \beta_{i,t}^{\prime} f_{t} + \epsilon_{i,t}, \quad t = 1, 2, \dots, T. $

for each stock at time t, use 60-month data to get beta

### Time Varying Beta

In [42]:
group = list(np.unique(data.variable))
# data[data.variable==group[0]] # restrict the data within a specific group

In [43]:
for g in range(len(group)):
    a = data[data.variable==group[g]][['date', 'variable']]
    b = RollingOLS.from_formula('exret ~ mktrf + hml + smb', data=data[data.variable==group[g]], window=60).fit().params

    if g == 0:
        df = pd.concat([a, b], axis=1)
    else:
        df = pd.concat([df, pd.concat([a, b], axis=1)], axis=0)

In [44]:
df.columns = ['date', 'variable', 'alpha', 'beta_mkt', 'beta_hml', 'beta_smb']

In [45]:
ff_beta = pd.merge(data, df, on=['date', 'variable'], how='left')

In [46]:
ff_beta

,date,variable,value,mktrf,hml,smb,rf,exret,alpha,beta_mkt,beta_hml,beta_smb
0,1926-06-30,s1b1_vwret,0.0583,0.0289,-0.0239,-0.0255,0.0022,0.0561,NaN,NaN,NaN,NaN
1,1926-07-31,s1b1_vwret,-0.0202,0.0264,0.0381,-0.0114,0.0025,-0.0227,NaN,NaN,NaN,NaN
2,1926-08-31,s1b1_vwret,-0.0483,0.0038,0.0005,-0.0136,0.0023,-0.0506,NaN,NaN,NaN,NaN
3,1926-09-30,s1b1_vwret,-0.0936,-0.0327,0.0082,-0.0014,0.0032,-0.0968,NaN,NaN,NaN,NaN
4,1926-10-31,s1b1_vwret,0.0559,0.0254,-0.0061,-0.0011,0.0031,0.0528,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
29545,2024-07-31,s5b5_vwret,0.0094,0.016,-0.011,-0.0349,0.0048,0.0046,0.000372,1.112184,0.857356,0.203462
29546,2024-08-31,s5b5_vwret,-0.0053,0.0172,-0.0277,-0.0013,0.004,-0.0093,0.000623,1.112898,0.869368,0.198076
29547,2024-09-30,s5b5_vwret,0.0589,-0.01,0.0086,-0.0099,0.0039,0.055,0.001232,1.105114,0.878293,0.189376
29548,2024-10-31,s5b5_vwret,0.1131,0.0649,0.0015,0.0446,0.004,0.1091,0.001324,1.107187,0.880415,0.205725


### Fama-Macbeth Regression

$$

\begin{align*}
r_{i,t}^{e} &= \beta_{i,t}^{\prime} \lambda_{t} + \alpha_{i,t}, \quad i = 1, 2, \dots, N \quad \text{for each } t. \\
\\
\hat{\lambda}_{FM} &= \frac{1}{T} \sum_{t=1}^{T} \hat{\lambda}_{t}, \quad \hat{\alpha}_{i,FM} = \frac{1}{T} \sum_{t=1}^{T} \hat{\alpha}_{i,t} \\
\\
\text{Var}( \hat{\lambda}_{FM} ) &= \frac{1}{T^2} \text{Var} \left( \sum_{t=1}^{T} \hat{\lambda}_{t} \right) = \frac{1}{T} \text{Var}( \hat{\lambda}_{t} ) = \frac{1}{T^2} \sum_{t=1}^{T} ( \hat{\lambda}_{t} - \hat{\lambda}_{FM} )^2 \\
\\
\text{Var}( \hat{\alpha}_{i,FM} ) &= \text{Var} \left( \frac{1}{T} \sum_{t=1}^{T} \hat{\alpha}_{i,t} \right) = \frac{1}{T} \text{Var}( \hat{\alpha}_{i,t} ) = \frac{1}{T^2} \sum_{t=1}^{T} ( \hat{\alpha}_{i,t} - \hat{\alpha}_{i,FM} )^2
\end{align*}

$$

### Cross sectional regression for each t

In [47]:
ff_beta = ff_beta[~ff_beta.isna().any(axis=1)] # drop rows with NA values
time = np.unique(ff_beta.date)

In [48]:
for t in range(len(time)): # for each time t, run cross-sectional regression
    result = pd.DataFrame(smf.ols('exret ~ beta_mkt + beta_hml + beta_smb - 1', data=ff_beta[ff_beta.date==time[t]]).fit().params)
    result.columns = [time[t]]

    resid = pd.DataFrame(smf.ols(formula='exret ~ beta_mkt + beta_hml + beta_smb - 1', data=ff_beta[ff_beta.date==time[t]]).fit().resid)
    resid = pd.concat([ff_beta[ff_beta.date==time[t]][['variable', 'date']], resid], axis=1)

    if t == 0:
        cs_res = result
        alpha = resid
    else:
        cs_res = pd.concat([cs_res, result], axis=1)
        alpha = pd.concat([alpha, resid], axis=0)

In [49]:
cs_res = cs_res.T
cs_res

,beta_mkt,beta_hml,beta_smb
1931-05-31,0.168256,0.109596,-0.080115
1931-06-30,-0.075835,-0.006066,0.010006
1931-07-31,-0.004043,0.022235,-0.024958
1931-08-31,-0.309051,-0.039576,0.015005
1931-09-30,0.085613,0.026900,-0.026907
...,...,...,...
2024-07-31,0.018719,-0.014633,-0.031369
2024-08-31,0.023514,-0.030537,-0.010601
2024-09-30,-0.014436,0.011975,0.005699
2024-10-31,0.071982,0.011893,0.028915


In [50]:
alpha.head()

,variable,date,0
59,s1b1_vwret,1931-05-31,0.066952
1241,s1b2_vwret,1931-05-31,-0.046646
2423,s1b3_vwret,1931-05-31,0.007377
3605,s1b4_vwret,1931-05-31,0.007281
4787,s1b5_vwret,1931-05-31,-0.002734


### FM coefficients

$$

\hat{\lambda}_{FM} = \frac{1}{T} \sum_{t=1}^{T} \hat{\lambda}_{t}

$$

In [51]:
## estimates

estimates = pd.DataFrame(cs_res.mean()) # Time-series average will be the resulting coefficient
estimates.columns = ['coefficients']
estimates * 12 # annualized

,coefficients
beta_mkt,0.083182
beta_hml,0.046706
beta_smb,0.021796


$$

Var( \hat{\lambda}_{FM} ) = \frac{1}{T^2} Var \left( \sum_{t=1}^{T} \hat{\lambda}_{t} \right) = \frac{1}{T} Var( \hat{\lambda}_{t} ) = \frac{1}{T^2} \sum_{t=1}^{T} ( \hat{\lambda}_{t} - \hat{\lambda}_{FM} )^2

$$

In [52]:
## standard errors 

variance = np.full(3, np.nan)

for i in range(3):
    variance[i] = (sum((cs_res.iloc[:, i] - cs_res.iloc[:, i].mean())**2) / (len(cs_res)**2)) ** 0.5

In [53]:
variance = pd.DataFrame(variance)
variance.index=['beta_mkt', 'beta_hml', 'beta_smb']

In [54]:
variance.columns = ['s.e']

In [55]:
result = pd.concat([estimates, variance], axis=1)
result['t' ] = result['coefficients'] / result['s.e']
result

,coefficients,s.e,t
beta_mkt,0.006932,0.001579,4.390247
beta_hml,0.003892,0.001113,3.497316
beta_smb,0.001816,0.000993,1.829369


### Alpha

$$

\hat{\alpha}_{i,FM} = \frac{1}{T} \sum_{t=1}^{T} \hat{\alpha}_{i,t}

$$

Check if alpha is collctively zero

In [56]:
alpha_fm = alpha.groupby('variable').mean()
alpha_fm = alpha_fm.reset_index()

In [60]:
alpha_fm.columns = ['variable', 'date', 'alpha_fm']

In [67]:
alpha.columns = ['variable', 'date', 'alpha']

In [83]:
variance = alpha.pivot_table(index='date', columns='variable', values='alpha')

In [84]:
variance = np.cov(np.asmatrix(variance.values).T)
variance.shape

(25, 25)

In [80]:
ndates = len(alpha['date'].unique())
ndates

1123

In [86]:
variance = variance / ndates

### Joint Alpha Test

$$

\hat{\alpha}_{i,FM}^{\prime} \cdot Var(\hat{\alpha}_{i,FM})^{-1} \hat{\alpha}_{i,FM} \sim \chi_{N-K}^2

$$

In [87]:
al = np.asmatrix(alpha_fm['alpha_fm'].values)
al

matrix([[-0.00360376, -0.00279436,  0.00014327,  0.00190912,  0.00242181,
         -0.00099228,  0.00086199,  0.00073096,  0.00121535,  0.0005953 ,
          0.00018696,  0.00101034,  0.00049961,  0.00063731, -0.00036468,
          0.00096107,  0.00026079,  0.00022033,  0.00045319, -0.00101334,
          0.00147457, -0.00020605,  0.00045514, -0.00227432, -0.00100482]])

In [89]:
chisq = al @ np.linalg.inv(variance) @ al.T

In [90]:
chisq

matrix([[87.05506229]])

In [92]:
pval = 1 - stats.chi2(25 - 3).cdf(chisq)

In [93]:
pval

array([[1.08115961e-09]])

In [94]:
pval < 0.05

array([[ True]])

## Method 2. linearmodels package

In [102]:
mulidx_ff_beta = ff_beta.set_index(['variable', 'date'], inplace=False)

In [106]:
mulidx_ff_beta = mulidx_ff_beta[['exret', 'beta_mkt', 'beta_hml', 'beta_smb']]

In [104]:
dependent = mulidx_ff_beta['exret']
exog = mulidx_ff_beta[['beta_mkt', 'beta_hml', 'beta_smb']]

In [ ]:
lm_model = lm.panel.model.FamaMacBeth(dependent, exog)
lm_result = lm_model.from_formula('exret ~ beta_mkt + beta_hml + beta_smb - 1', data=mulidx_ff_beta).fit()


In [110]:
lm_result

Dep. Variable:,exret,R-squared:,0.0170
Estimator:,FamaMacBeth,R-squared (Between):,0.9794
No. Observations:,28075,R-squared (Within):,0.0008
Date:,"Sun, Nov 09 2025",R-squared (Overall):,0.0170
Time:,11:22:56,Log-likelihood,3.259e+04
Cov. Estimator:,Fama-MacBeth Standard Cov,,
,,F-statistic:,161.71
Entities:,25,P-value,0.0000
Avg Obs:,1123.0,Distribution:,"F(3,28072)"
Min Obs:,1123.0,,
Max Obs:,1123.0,F-statistic (robust):,8.8568


Confirmed: two methods give the same results

## Newey West standard errors

In [116]:
lm_model = lm.panel.model.FamaMacBeth(
    dependent, 
    exog,
    )
lm_result = lm_model.fit(
    cov_type='kernel',
    bandwidth=2,
    kernel='newey-west',
)


In [117]:
lm_result

Dep. Variable:,exret,R-squared:,0.0170
Estimator:,FamaMacBeth,R-squared (Between):,0.9794
No. Observations:,28075,R-squared (Within):,0.0008
Date:,"Sun, Nov 09 2025",R-squared (Overall):,0.0170
Time:,13:08:47,Log-likelihood,3.259e+04
Cov. Estimator:,Fama-MacBeth Kernel Cov,,
,,F-statistic:,161.71
Entities:,25,P-value,0.0000
Avg Obs:,1123.0,Distribution:,"F(3,28072)"
Min Obs:,1123.0,,
Max Obs:,1123.0,F-statistic (robust):,7.5741
